# Feather v2 quick baseline - real training run

Trains the shipped `configs/feather_20M_simple.json` configuration and reports only
values measured during the run.

**Size:** the config measures **20,696,188 parameters (20.70M)** against a 20M
target. The originally specified `dim=384` measures 23.12M at `n_blocks=2` and
82.36M at `n_blocks=8`, so `dim=352` was chosen to land near the target. Verify
either number yourself:

```python
!python -m feather_v2.model --config configs/feather_20M_simple.json --count-params
```

**What this notebook does not do:** it contains no target loss, no target
throughput, no predicted hardware numbers, and no recall score. The metric labelled
`StateStab` is `state_stability`, a cosine similarity between a position's hidden
state with and without a later suffix. It is not recall and not accuracy.

**Speed:** the defaults below are the small, fast configuration. 600 steps at
batch 2 x seq 128 is 256 tokens per step and projects to roughly 48 minutes;
batch 8 x seq 512 is 4096 tokens per step and projects to roughly 12 hours. Those
totals are projections from a measured 4.8 s/step at batch 2 x seq 128 on a
2-core laptop, not completed runs. An earlier specification claimed 10-20 minutes
for 600 steps at batch 8 x seq 512; that is not achievable on this hardware.

## 1. Environment

Checks the interpreter, installs the local package, and reports what codecarbon can
actually measure on this machine. GPU is not used; this is a CPU baseline.

In [ ]:
import json
import os
import platform
import subprocess
import sys

!pip -q install -e . codecarbon datasets matplotlib psutil

print("python  :", sys.version.split()[0])
print("platform:", platform.platform())
try:
    import torch

    print("torch   :", torch.__version__, "| threads:", torch.get_num_threads())
except Exception as exc:  # noqa: BLE001
    print("torch   : MISSING", exc)

from pathlib import Path

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print("root    :", ROOT)

In [ ]:
# Print the real parameter count instead of trusting the filename.
count = subprocess.run(
    [sys.executable, "-m", "feather_v2.model",
     "--config", "configs/feather_20M_simple.json", "--count-params"],
    cwd=ROOT, text=True, capture_output=True,
)
print(count.stdout or count.stderr)

In [ ]:
# Confirm the package under test is the local one, not a stale wheel.
import feather_v2

print("feather_v2 :", feather_v2.__file__)
assert Path(feather_v2.__file__).resolve().is_relative_to(ROOT), (
    "importing feather_v2 from outside the working directory"
)
print("version    :", getattr(feather_v2, "__version__", "n/a"))

## 2. Run the training script

The defaults match the quick baseline: 600 steps, batch 2, sequence length 128.
That is 256 tokens per step and projects to roughly 48 minutes on a 2-core
machine, which fits a Kaggle CPU session comfortably.

For the full-size run, raise both together, since cost scales with
`batch-size * seq-len`:

```python
# full run, 4096 tokens per step, projects to roughly 12 hours
!python kaggle/train_20M_simple.py --steps 600 --batch-size 8 --seq-len 512 \
    --time-budget-hours 12
```

`--time-budget-hours` is a guard, not a target. The script measures the first three
timed steps, projects the finish time, and stops early with a clear message if the
projection exceeds the budget, so a long run cannot silently occupy a session for a
day.

The first run downloads a Wikipedia slice through `datasets`, which needs network
access. Without it, pass `--allow-local-text` to fall back to repository text; the
artifact is then flagged as a local fallback rather than a real corpus.

In [ ]:
STEPS = 600
BATCH = 2
SEQ = 128
BUDGET_HOURS = 10.0

# Cost scales with BATCH * SEQ. 2 x 128 = 256 tokens/step, about 48 min projected.
# 8 x 512 = 4096 tokens/step, about 12 h projected.
cmd = [
    sys.executable,
    "kaggle/train_20M_simple.py",
    "--steps", str(STEPS),
    "--batch-size", str(BATCH),
    "--seq-len", str(SEQ),
    "--report-every", "50",
    "--out", "benchmark_20M_simple.json",
    "--time-budget-hours", str(BUDGET_HOURS),
]
print(" ".join(cmd), flush=True)
result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(result.stdout[-6000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
    raise SystemExit(f"training script failed with code {result.returncode}")

## 3. Read the results

`benchmark_20M_simple.json` holds the measurements, the per-step history, and the
authenticity gates. The gates test whether the run is genuine (finite and varying
loss, weights actually updated, tokens accounted for, energy actually observed).
They are not target checks: a short run can fail `loss_decreased` honestly, because
4 steps cannot show a trend.

A `VERDICT: FAIL` line with only `loss_decreased` failing is the expected result of
a short smoke run, not a broken model.

In [ ]:
import json as _json
from pathlib import Path as _Path

report_path = _Path("benchmark_20M_simple.json")
if not report_path.exists():
    raise SystemExit(
        "benchmark_20M_simple.json was not written, so there is nothing to report. "
        "Read the output of the previous cell first."
    )

report = _json.loads(report_path.read_text(encoding="utf-8"))
summary = report["summary"]

print("model      :", report["model"]["parameters"], "parameters")
print("corpus     :", report["corpus"].get("tokens"), "tokens from",
      [s["dataset"] for s in report["corpus"]["sources"] if s.get("ok")])
print("tokens     :", summary["tokens_trained"])
print("loss       :", summary.get("loss_start_mean"), "->", summary.get("loss_end_mean"))
print("tps        :", summary.get("training_tps_median"))
print("step secs  :", summary.get("step_seconds_median"))
print("peak rss   :", summary.get("rss_peak_mb"), "MB")
print("energy     :", summary.get("energy_total_j"), "J from", summary.get("energy_source"))
print("stability  :", summary.get("state_stability_final"), "(not a recall score)")
print()
for gate in report["gates"]:
    mark = "PASS" if gate["ok"] else ("SKIP" if gate["ok"] is None else "FAIL")
    print(f"[{mark}] {gate['check']}: {gate['detail']}")
print()
print("gates passed:", summary["gates_passed"], " failed:", summary["gates_failed"])

## 4. Plots

All five figures are generated from the measured run only.

In [ ]:
from IPython.display import Image, display
from pathlib import Path as _P

figures = [
    "loss_20M_simple.png",
    "tps_20M_simple.png",
    "ram_20M_simple.png",
    "energy_20M_simple.png",
    "recall_20M_simple.png",
]
image_dir = _P("docs/images")
for name in figures:
    path = image_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))
    else:
        print(f"{name}: not generated by this run")